In [ ]:
%pip install neo4j

# Docker ง่ายสุด
docker run \
  --name thai-const-graph \
  -p 7474:7474 -p 7687:7687 \
  -e NEO4J_AUTH=neo4j/password \
  neo4j:latest

## Cell 1: Connect

In [ ]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    "bolt://localhost:7687",
    auth=("neo4j", "password")
)

with driver.session() as session:
    result = session.run("RETURN 'connected' AS status")
    print(result.single()["status"])

## Cell 2: Load JSON

In [ ]:
import json
from pathlib import Path

DATA_PATH = Path("output/sections_graph.json")

with open(DATA_PATH, encoding="utf-8") as f:
    sections = json.load(f)

print(f"Loaded {len(sections)} sections")

## Cell 3: Constraints

In [ ]:
constraints = [
    "CREATE CONSTRAINT IF NOT EXISTS FOR (c:Constitution) REQUIRE c.doc_id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (ch:Chapter) REQUIRE ch.chapter_id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (s:Section) REQUIRE s.section_id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (i:Item) REQUIRE i.item_id IS UNIQUE",
]

with driver.session() as session:
    for q in constraints:
        session.run(q)

print("Constraints created")

## Cell 4: Insert Constitution nodes

In [ ]:
# deduplicate by doc_id
constitutions = {}
for s in sections:
    doc_id = s["doc_id"]
    if doc_id not in constitutions:
        constitutions[doc_id] = {
            "doc_id": doc_id,
            "doc_type": s["doc_type"],
            "year_th": s["year_th"],
            "year_ce": s["year_ce"],
            "name_short": s["name_short"],
            "era": s["era"],
            "regime_type": s["regime_type"],
            "parent_doc_id": s.get("parent_doc_id"),
        }

query = """
UNWIND $rows AS row
MERGE (c:Constitution {doc_id: row.doc_id})
SET c.doc_type    = row.doc_type,
    c.year_th     = row.year_th,
    c.year_ce     = row.year_ce,
    c.name_short  = row.name_short,
    c.era         = row.era,
    c.regime_type = row.regime_type
"""

with driver.session() as session:
    session.run(query, rows=list(constitutions.values()))

print(f"Inserted {len(constitutions)} Constitution nodes")

## Cell 5: Insert Chapter nodes + BELONGS_TO edges

In [ ]:
chapters = {}
for s in sections:
    key = (s["doc_id"], s["chapter_number"])
    if key not in chapters:
        chapter_id = f"{s['doc_id']}_c_{s['chapter_number']}"
        chapters[key] = {
            "chapter_id": chapter_id,
            "doc_id": s["doc_id"],
            "chapter_number": s["chapter_number"],
            "chapter_title": s.get("chapter_title"),
        }

query = """
UNWIND $rows AS row
MERGE (ch:Chapter {chapter_id: row.chapter_id})
SET ch.chapter_number = row.chapter_number,
    ch.chapter_title  = row.chapter_title,
    ch.doc_id         = row.doc_id
WITH ch, row
MATCH (c:Constitution {doc_id: row.doc_id})
MERGE (ch)-[:BELONGS_TO]->(c)
"""

with driver.session() as session:
    session.run(query, rows=list(chapters.values()))

print(f"Inserted {len(chapters)} Chapter nodes + BELONGS_TO edges")

## Cell 6: Insert Section nodes + PART_OF edges

In [ ]:
section_rows = []
for s in sections:
    section_rows.append({
        "section_id":         s["section_id"],
        "doc_id":             s["doc_id"],
        "section_number":     s["section_number"],
        "section_role":       s["section_role"],
        "change_mode":        s["change_mode"],
        "chapter_id":         f"{s['doc_id']}_c_{s['chapter_number']}",
        "sub_section_number": s.get("sub_section_number"),
        "sub_section_title":  s.get("sub_section_title"),
        "target_section_no":  s.get("target_section_no"),
        "parent_doc_id":      s.get("parent_doc_id"),
        "text":               s["text"],
    })

# insert in batches of 500
query = """
UNWIND $rows AS row
MERGE (s:Section {section_id: row.section_id})
SET s.doc_id             = row.doc_id,
    s.section_number     = row.section_number,
    s.section_role       = row.section_role,
    s.change_mode        = row.change_mode,
    s.sub_section_number = row.sub_section_number,
    s.sub_section_title  = row.sub_section_title,
    s.target_section_no  = row.target_section_no,
    s.parent_doc_id      = row.parent_doc_id,
    s.text               = row.text
WITH s, row
MATCH (ch:Chapter {chapter_id: row.chapter_id})
MERGE (s)-[:PART_OF]->(ch)
"""

BATCH = 500
with driver.session() as session:
    for i in range(0, len(section_rows), BATCH):
        session.run(query, rows=section_rows[i:i+BATCH])

print(f"Inserted {len(section_rows)} Section nodes + PART_OF edges")

## Cell 7: Insert Item nodes + HAS_ITEM edges

In [ ]:
item_rows = []
for s in sections:
    for item in s.get("items", []):
        item_rows.append({
            "item_id":    item["item_id"],
            "section_id": s["section_id"],
            "number":     item["number"],
            "text":       item["text"],
        })

query = """
UNWIND $rows AS row
MERGE (i:Item {item_id: row.item_id})
SET i.number     = row.number,
    i.text       = row.text,
    i.section_id = row.section_id
WITH i, row
MATCH (s:Section {section_id: row.section_id})
MERGE (s)-[:HAS_ITEM]->(i)
"""

BATCH = 500
with driver.session() as session:
    for i in range(0, len(item_rows), BATCH):
        session.run(query, rows=item_rows[i:i+BATCH])

print(f"Inserted {len(item_rows)} Item nodes + HAS_ITEM edges")

## Cell 8: Create edges

In [ ]:
# AMENDS: Constitution -> Constitution
amends_rows = [
    {"doc_id": doc_id, "parent_doc_id": meta["parent_doc_id"]}
    for doc_id, meta in constitutions.items()
    if meta.get("parent_doc_id")
]

with driver.session() as session:
    session.run("""
        UNWIND $rows AS row
        MATCH (a:Constitution {doc_id: row.doc_id})
        MATCH (b:Constitution {doc_id: row.parent_doc_id})
        MERGE (a)-[:AMENDS]->(b)
    """, rows=amends_rows)

print(f"AMENDS edges: {len(amends_rows)}")

In [ ]:
# AMENDS_SECTION: wrapper Section -> target Section
amends_sec_rows = [
    {
        "src": s["section_id"],
        "tgt": f"{s['parent_doc_id']}_s_{s['target_section_no']}",
        "change_mode": s["change_mode"],
    }
    for s in sections
    if s["section_role"] == "wrapper" and s.get("target_section_no") and s.get("parent_doc_id")
]

with driver.session() as session:
    session.run("""
        UNWIND $rows AS row
        MATCH (a:Section {section_id: row.src})
        MATCH (b:Section {section_id: row.tgt})
        MERGE (a)-[r:AMENDS_SECTION]->(b)
        SET r.change_mode = row.change_mode
    """, rows=amends_sec_rows)

print(f"AMENDS_SECTION edges: {len(amends_sec_rows)}")

In [ ]:
# REFERENCES: Section -> Section (section-level)
sec_ref_rows = [
    {"src": s["section_id"], "tgt": ref["section_id"]}
    for s in sections
    for ref in s.get("references", [])
]

# REFERENCES: Item -> Section (item-level)
item_ref_rows = [
    {"src": item["item_id"], "tgt": ref["section_id"]}
    for s in sections
    for item in s.get("items", [])
    for ref in item.get("references", [])
]

with driver.session() as session:
    # section-level refs
    BATCH = 500
    for i in range(0, len(sec_ref_rows), BATCH):
        session.run("""
            UNWIND $rows AS row
            MATCH (a:Section {section_id: row.src})
            MATCH (b:Section {section_id: row.tgt})
            MERGE (a)-[:REFERENCES]->(b)
        """, rows=sec_ref_rows[i:i+BATCH])

    # item-level refs
    for i in range(0, len(item_ref_rows), BATCH):
        session.run("""
            UNWIND $rows AS row
            MATCH (a:Item {item_id: row.src})
            MATCH (b:Section {section_id: row.tgt})
            MERGE (a)-[:REFERENCES]->(b)
        """, rows=item_ref_rows[i:i+BATCH])

print(f"REFERENCES edges (section-level): {len(sec_ref_rows)}")
print(f"REFERENCES edges (item-level)   : {len(item_ref_rows)}")

## Cell 9: Verify counts

In [ ]:
queries = {
    "Constitution" : "MATCH (n:Constitution) RETURN count(n) AS c",
    "Chapter"      : "MATCH (n:Chapter) RETURN count(n) AS c",
    "Section"      : "MATCH (n:Section) RETURN count(n) AS c",
    "Item"         : "MATCH (n:Item) RETURN count(n) AS c",
    "AMENDS"       : "MATCH ()-[r:AMENDS]->() RETURN count(r) AS c",
    "BELONGS_TO"   : "MATCH ()-[r:BELONGS_TO]->() RETURN count(r) AS c",
    "PART_OF"      : "MATCH ()-[r:PART_OF]->() RETURN count(r) AS c",
    "HAS_ITEM"     : "MATCH ()-[r:HAS_ITEM]->() RETURN count(r) AS c",
    "AMENDS_SECTION": "MATCH ()-[r:AMENDS_SECTION]->() RETURN count(r) AS c",
    "REFERENCES"   : "MATCH ()-[r:REFERENCES]->() RETURN count(r) AS c",
}

with driver.session() as session:
    print("=== Neo4j counts ===")
    for label, q in queries.items():
        count = session.run(q).single()["c"]
        print(f"  {label:<20}: {count}")